In [1]:
import pandas as pd
import numpy as np

df=pd.read_csv("Day16_Student_Wellbeing_Survey.csv")
print("Shape:",df.shape)
display(df.head())
display(df.dtypes)
display(df.isna().sum().sort_values(ascending=False))
print("Duplicate rows:",df.duplicated().sum())

Shape: (600, 20)


,Student_ID,Age,Faculty,Year_of_Study,City,Accommodation,Scholarship,Part_Time_Job,Internet_Quality,Preferred_Study_Space,Weekly_Study_Hours,Average_Sleep_Hours,Daily_Screen_Time_Hours,Exercise_Days_Per_Week,Commute_Time_Minutes,Stress_Score,Academic_Readiness_Score,Overall_Satisfaction,Monthly_Discretionary_Spending,Social_Activity_Hours_Per_Week
0,STU0001,21,Data Science,3,Bengaluru,Hostel,No,Yes,Good,Café,15.2,5.7,3.2,3,11.8,5.4,63.1,3.2,5131.0,7.8
1,STU0002,21,Data Science,4,Pune,Hostel,Yes,No,Poor,Library,17.6,7.7,3.5,4,20.3,2.1,74.2,3.8,3508.0,9.2
2,STU0003,22,Life Sciences,2,Chandigarh,Home,Yes,No,Poor,Home,18.9,6.4,2.8,1,15.0,5.2,77.3,4.0,4584.0,7.7
3,STU0004,18,Business,4,Bengaluru,Hostel,Yes,Yes,Average,Café,19.5,6.7,3.0,5,12.1,4.7,67.0,4.0,5456.0,5.3
4,STU0005,20,Business,3,Jammu,Shared Apartment,Yes,No,Good,Home,18.4,7.3,3.0,1,30.2,5.4,65.6,3.2,11732.0,15.9


Student_ID                            str
Age                                 int64
Faculty                               str
Year_of_Study                       int64
City                                  str
Accommodation                         str
Scholarship                           str
Part_Time_Job                         str
Internet_Quality                      str
Preferred_Study_Space                 str
Weekly_Study_Hours                float64
Average_Sleep_Hours               float64
Daily_Screen_Time_Hours           float64
Exercise_Days_Per_Week              int64
Commute_Time_Minutes              float64
Stress_Score                      float64
Academic_Readiness_Score          float64
Overall_Satisfaction              float64
Monthly_Discretionary_Spending    float64
Social_Activity_Hours_Per_Week    float64
dtype: object

Student_ID                        0
Age                               0
Faculty                           0
Year_of_Study                     0
City                              0
Accommodation                     0
Scholarship                       0
Part_Time_Job                     0
Internet_Quality                  0
Preferred_Study_Space             0
Weekly_Study_Hours                0
Average_Sleep_Hours               0
Daily_Screen_Time_Hours           0
Exercise_Days_Per_Week            0
Commute_Time_Minutes              0
Stress_Score                      0
Academic_Readiness_Score          0
Overall_Satisfaction              0
Monthly_Discretionary_Spending    0
Social_Activity_Hours_Per_Week    0
dtype: int64

Duplicate rows: 0


Mean, Median, Mode and Variability

In [2]:
vars5=['Weekly_Study_Hours', 'Average_Sleep_Hours', 'Daily_Screen_Time_Hours', 'Stress_Score', 'Academic_Readiness_Score']
for c in vars5:
    df[c]=pd.to_numeric(df[c],errors="coerce")

rows=[]
for c in vars5:
    s=df[c].dropna()
    rows.append([c,s.mean(),s.median(),", ".join(map(str,s.mode().tolist())),s.max()-s.min(),s.var(),s.std(),s.quantile(.25),s.quantile(.75),s.quantile(.75)-s.quantile(.25)])
stats=pd.DataFrame(rows,columns=["Variable","Mean","Median","Mode","Range","Variance","Std_Dev","Q1","Q3","IQR"])
display(stats)

print("Greatest variability by standard deviation:",stats.loc[stats.Std_Dev.idxmax(),"Variable"])
print("Greatest variability by variance:",stats.loc[stats.Variance.idxmax(),"Variable"])

,Variable,Mean,Median,Mode,Range,Variance,Std_Dev,Q1,Q3,IQR
0,Weekly_Study_Hours,15.691000,15.40,"13.1, 14.6, 15.3, 17.1",30.0,19.217047,4.383725,13.000,18.425,5.425
1,Average_Sleep_Hours,6.997500,7.00,7.0,5.0,0.703383,0.838679,6.475,7.600,1.125
2,Daily_Screen_Time_Hours,4.503000,4.20,3.5,11.2,3.469006,1.862527,3.200,5.400,2.200
3,Stress_Score,4.464500,4.50,"4.7, 4.9",8.4,2.782494,1.668081,3.300,5.700,2.400
4,Academic_Readiness_Score,71.773167,71.65,"71.1, 71.4, 78.0",56.1,93.981967,9.694430,65.200,78.025,12.825


Greatest variability by standard deviation: Academic_Readiness_Score
Greatest variability by variance: Academic_Readiness_Score


IQR Outlier Detection

In [3]:
outlier_vars=['Weekly_Study_Hours', 'Daily_Screen_Time_Hours', 'Commute_Time_Minutes', 'Monthly_Discretionary_Spending']
out_rows=[]
out_masks={}
for c in outlier_vars:
    s=df[c].dropna()
    q1,q3=s.quantile(.25),s.quantile(.75)
    iqr=q3-q1
    lo,hi=q1-1.5*iqr,q3+1.5*iqr
    mask=(df[c]<lo)|(df[c]>hi)
    out_masks[c]=mask
    out_rows.append([c,q1,q3,iqr,lo,hi,int(mask.sum())])

outlier_table=pd.DataFrame(out_rows,columns=["Variable","Q1","Q3","IQR","Lower_Bound","Upper_Bound","Outlier_Count"])
display(outlier_table)

,Variable,Q1,Q3,IQR,Lower_Bound,Upper_Bound,Outlier_Count
0,Weekly_Study_Hours,13.000,18.425,5.425,4.8625,26.5625,8
1,Daily_Screen_Time_Hours,3.200,5.400,2.200,-0.1000,8.7000,18
2,Commute_Time_Minutes,13.375,31.525,18.150,-13.8500,58.7500,4
3,Monthly_Discretionary_Spending,4113.000,7808.500,3695.500,-1430.2500,13351.7500,20


Mean and Median Before/After Outlier Removal

In [4]:
m=out_masks["Weekly_Study_Hours"]
before_mean=df["Weekly_Study_Hours"].mean()
before_median=df["Weekly_Study_Hours"].median()
after=df.loc[~m,"Weekly_Study_Hours"].dropna()

comparison=pd.DataFrame({
    "Statistic":["Mean","Median"],
    "Before":[before_mean,before_median],
    "After":[after.mean(),after.median()]
})
display(comparison)

,Statistic,Before,After
0,Mean,15.691,15.602872
1,Median,15.400,15.350000


Probability Events

In [5]:
excol='Exercise_Days_Per_Week'
df[excol]=pd.to_numeric(df[excol],errors="coerce")
pdat=df.dropna(subset=["Part_Time_Job","Stress_Score","Scholarship",excol]).copy()

A=pdat["Part_Time_Job"].astype(str).str.strip().str.lower().eq("yes")
B=pdat["Stress_Score"]>=7
C=pdat["Scholarship"].astype(str).str.strip().str.lower().eq("yes")
D=pdat[excol]>=3

pA=A.mean(); pB=B.mean(); pC=C.mean(); pD=D.mean()
pAB=(A&B).mean(); pAoB=(A|B).mean()
pAgB=(A&B).sum()/B.sum(); pBgA=(A&B).sum()/A.sum()

print("Sample size used:",len(pdat))
print("P(A) =",pA)
print("P(B) =",pB)
print("P(C) =",pC)
print("P(D) =",pD)
print("P(A or B) =",pAoB)
print("P(A and B) =",pAB)
print("P(A | B) =",pAgB)
print("P(B | A) =",pBgA)

Sample size used: 600
P(A) = 0.25333333333333335
P(B) = 0.075
P(C) = 0.30666666666666664
P(D) = 0.5933333333333334
P(A or B) = 0.27666666666666667
P(A and B) = 0.051666666666666666
P(A | B) = 0.6888888888888889
P(B | A) = 0.20394736842105263


Mutual Exclusivity: Year 1 vs Year 4

In [6]:
y1=df["Year_of_Study"]==1
y4=df["Year_of_Study"]==4
print("Intersection count:",(y1&y4).sum())
print("Mutually exclusive:",not (y1&y4).any())

Intersection count: 0
Mutually exclusive: True


Independence of A and B

In [7]:
lhs=pAB
rhs=pA*pB
print("P(A and B) =",lhs)
print("P(A) × P(B) =",rhs)
print("Difference =",lhs-rhs)
print("Approximately independent:",np.isclose(lhs,rhs,atol=0.01))

P(A and B) = 0.051666666666666666
P(A) × P(B) = 0.019
Difference = 0.03266666666666666
Approximately independent: False


Bayes' Theorem

In [8]:
pBnotA=(B&~A).sum()/(~A).sum()
bayes=(pBgA*pA)/(pBgA*pA+pBnotA*(1-pA))

print("P(B | A) =",pBgA)
print("P(B | not A) =",pBnotA)
print("P(A) =",pA)
print("P(not A) =",1-pA)
print("Bayes P(A | B) =",bayes)
print("Direct P(A | B) =",pAgB)
print("Agreement:",np.isclose(bayes,pAgB))

P(B | A) = 0.20394736842105263
P(B | not A) = 0.03125
P(A) = 0.25333333333333335
P(not A) = 0.7466666666666666
Bayes P(A | B) = 0.6888888888888889
Direct P(A | B) = 0.6888888888888889
Agreement: True


Normal Distribution and Z-Scores

In [9]:
ars=df["Academic_Readiness_Score"].dropna()
mu=ars.mean()
sigma=ars.std()

max_idx=df["Academic_Readiness_Score"].idxmax()
min_idx=df["Academic_Readiness_Score"].idxmin()
max_score=df.loc[max_idx,"Academic_Readiness_Score"]
min_score=df.loc[min_idx,"Academic_Readiness_Score"]

z_max=(max_score-mu)/sigma
z_min=(min_score-mu)/sigma

print("Mean =",mu)
print("Standard deviation =",sigma)
print("Highest score =",max_score," Z =",z_max)
print("Lowest score =",min_score," Z =",z_min)

Mean = 71.77316666666667
Standard deviation = 9.694429667762524
Highest score = 98.0  Z = 2.705350828481124
Lowest score = 41.9  Z = -3.081477476287824


68–95–99.7 Empirical Rule

In [10]:
for k,expected in [(1,.68),(2,.95),(3,.997)]:
    lo,hi=mu-k*sigma,mu+k*sigma
    actual=((ars>=lo)&(ars<=hi)).mean()
    print(f"±{k} SD: expected={expected:.1%}, actual sample={actual:.1%}")

±1 SD: expected=68.0%, actual sample=69.5%
±2 SD: expected=95.0%, actual sample=95.2%
±3 SD: expected=99.7%, actual sample=99.8%
